# CV Deep Research Agent
## Career Positioning through Deep Research

This agent goes beyond a basic CV review. It takes a CV, a target role, and optionally a company name, then:

1. Extracts CV content from PDF or text
2. Checks ATS compliance: section headers, contact info, formatting, action verbs, metrics
3. Researches the target role online: in-demand skills, common keywords, market trends
4. Researches the target company: culture, tech stack, hiring focus
5. Analyzes gaps: compares the CV against role requirements and market demand
6. Generates a comprehensive report with rewrite suggestions, keyword additions, and interview prep

In [ ]:
from agents import Agent, WebSearchTool, Runner, trace, gen_trace_id, function_tool, input_guardrail, GuardrailFunctionOutput
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from pypdf import PdfReader
import asyncio
import json
import os
import re
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)

## Tools Definitions

In [ ]:
@function_tool
def extract_cv_text(file_path: str) -> dict:
    """Read a CV/resume from a PDF or text file and return its text content."""
    try:
        if file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            text = ""
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t
        else:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
        return {"success": True, "text": text, "word_count": len(text.split())}
    except FileNotFoundError:
        return {"success": False, "error": f"File not found: {file_path}"}

In [ ]:
@function_tool
def check_ats_formatting(cv_text: str) -> dict:
    """Run ATS compliance checks on CV text: section headers, length, contact info, action verbs, quantified achievements."""
    results = {}

    # Standard section headers
    standard_sections = ["summary", "experience", "education", "skills", "contact"]
    results["found_sections"] = [s for s in standard_sections if s in cv_text.lower()]
    results["missing_sections"] = [s for s in standard_sections if s not in cv_text.lower()]

    # Word count
    word_count = len(cv_text.split())
    results["word_count"] = word_count
    results["length_verdict"] = "too short" if word_count < 300 else "too long" if word_count > 1000 else "good"

    # Contact info
    results["has_email"] = bool(re.search(r'\b[\w.-]+@[\w.-]+\.\w+\b', cv_text))
    results["has_phone"] = bool(re.search(r'[\+\(]?[0-9][\d \-\(\)]{7,}\d', cv_text))

    # Action verbs
    action_verbs = ["led", "managed", "developed", "designed", "implemented", "achieved",
                    "increased", "reduced", "created", "launched", "built", "delivered",
                    "optimized", "streamlined", "mentored", "spearheaded", "orchestrated"]
    found_verbs = [v for v in action_verbs if v in cv_text.lower()]
    results["action_verbs_found"] = found_verbs
    results["action_verb_count"] = len(found_verbs)

    # Quantified achievements
    results["has_quantified_achievements"] = bool(re.search(r'\d+%|\$[\d,]+|\d+\+', cv_text))

    return results

In [ ]:
@function_tool
def check_keyword_match(cv_text: str, job_keywords: str) -> dict:
    """Compare CV text against comma-separated job requirement keywords and return a match score."""
    cv_lower = cv_text.lower()
    keywords = [kw.strip().lower() for kw in job_keywords.split(",") if kw.strip()]
    matched = [kw for kw in keywords if kw in cv_lower]
    missing = [kw for kw in keywords if kw not in cv_lower]
    score = round(len(matched) / len(keywords) * 100, 1) if keywords else 0
    return {"matched_keywords": matched, "missing_keywords": missing, "match_score_pct": score}

In [ ]:
@function_tool
def save_report(filename: str, report_content: str) -> dict:
    """Save the final CV deep research report to a markdown file."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(report_content)
    return {"saved": filename, "size_chars": len(report_content)}

## Structured Outputs

We define typed output schemas so each agent returns structured, predictable data.

In [ ]:
# Planner output

class ResearchItem(BaseModel):
    reason: str = Field(description="Why this search is important for the CV review.")
    query: str = Field(description="The search term to use for the web search.")


class ResearchPlan(BaseModel):
    role_searches: list[ResearchItem] = Field(description="Searches about the target role: skills, keywords, trends.")
    company_searches: list[ResearchItem] = Field(description="Searches about the target company (empty list if no company provided).")

In [ ]:
# Gap analysis output

class GapItem(BaseModel):
    area: str = Field(description="The gap area: skills, keywords, experience, metrics, or structure.")
    severity: str = Field(description="critical, major, or minor.")
    detail: str = Field(description="What is missing or weak.")
    fix: str = Field(description="Specific suggestion to close this gap.")


class GapAnalysis(BaseModel):
    overall_fit_score: int = Field(description="How well the CV fits the target role, 0-100.")
    gaps: list[GapItem] = Field(description="All gaps found between the CV and the target role/market.")
    strengths: list[str] = Field(description="What the CV already does well for this role.")

In [ ]:
# Final report output

class RewriteSuggestion(BaseModel):
    original: str = Field(description="The original bullet point or section from the CV.")
    improved: str = Field(description="The improved version with stronger impact and keywords.")
    reason: str = Field(description="Why this rewrite is better.")


class FinalReport(BaseModel):
    executive_summary: str = Field(description="2-3 sentence overview of the CV review findings.")
    ats_readiness_score: int = Field(description="ATS compliance score out of 100.")
    role_fit_score: int = Field(description="How well the CV matches the target role, 0-100.")
    market_insights: str = Field(description="Key findings from market research about the target role.")
    company_insights: str = Field(description="Key findings about the target company (or 'N/A' if none).")
    gap_summary: str = Field(description="Summary of the most critical gaps found.")
    rewrite_suggestions: list[RewriteSuggestion] = Field(description="At least 3 specific bullet point rewrites.")
    keywords_to_add: list[str] = Field(description="Keywords the candidate should weave into the CV.")
    interview_prep_topics: list[str] = Field(description="5+ topics the candidate should prepare for interviews.")
    thirty_day_action_plan: list[str] = Field(description="Ordered steps the candidate should take in the next 30 days.")

## Guardrails

A guardrail to reject vague requests that don't specify a CV file or target role.

In [ ]:
class InputCheckOutput(BaseModel):
    has_cv_file: bool = Field(description="Whether a CV file path was mentioned.")
    has_target_role: bool = Field(description="Whether a target job role was mentioned.")
    reason: str = Field(description="Explanation of what's missing, if anything.")


input_check_agent = Agent(
    name="Input Validator",
    instructions="Check if the user's message includes (1) a CV file path and (2) a target job role. Both are required.",
    output_type=InputCheckOutput,
    model="gpt-4o-mini",
)


@input_guardrail
async def validate_input(ctx, agent, message):
    """Reject requests that don't include a CV file and a target role."""
    result = await Runner.run(input_check_agent, message, context=ctx.context)
    output = result.final_output
    is_invalid = not output.has_cv_file or not output.has_target_role
    return GuardrailFunctionOutput(
        output_info={"check": output},
        tripwire_triggered=is_invalid,
    )

## Specialist Agents

Each agent has a focused role in the research pipeline

In [ ]:
# 1. Planner Agent — plans the research

HOW_MANY_ROLE_SEARCHES = 3
HOW_MANY_COMPANY_SEARCHES = 2

planner_agent = Agent(
    name="Research Planner",
    instructions=f"""You are a career research planner. Given a target job role and optionally a company name,
produce a research plan with:
- {HOW_MANY_ROLE_SEARCHES} web searches about the role (in-demand skills, common job keywords, salary trends, market outlook)
- {HOW_MANY_COMPANY_SEARCHES} web searches about the company (if provided; otherwise return an empty list for company_searches)

Make searches specific and current. Include the year 2025/2026 in queries for freshness.""",
    model="gpt-4o-mini",
    output_type=ResearchPlan,
)

In [ ]:
# 2. Search Agent — performs web searches

search_agent = Agent(
    name="Search Agent",
    instructions="""You are a research assistant. Given a search term, you search the web and
produce a concise summary of the results in 2-3 paragraphs (under 300 words).
Capture the main points. Write succinctly — no filler.
This will be consumed by an analyst, so capture the essence and skip any fluff.""",
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
# 3. ATS Reviewer Agent — runs the CV compliance tools

ats_reviewer_agent = Agent(
    name="ATS Reviewer",
    instructions="""You are an ATS compliance expert. Given a CV file path and optionally job keywords:
1. Use extract_cv_text to read the CV
2. Use check_ats_formatting to get compliance data
3. If job keywords are provided, use check_keyword_match

Return ALL the raw results so the analyst can use them. Include the full CV text in your response.""",
    tools=[extract_cv_text, check_ats_formatting, check_keyword_match],
    model="gpt-4o-mini",
)

In [ ]:
# 4. Gap Analyst Agent — compares CV vs research findings

gap_analyst_agent = Agent(
    name="Gap Analyst",
    instructions="""You are a career gap analyst. You receive:
- The full CV text and ATS check results
- Market research about the target role
- Company research (if available)

Analyze every gap between what the CV shows and what the market demands.
Be thorough — find at least 5 gaps. Rate each by severity.
Also identify the candidate's existing strengths.""",
    model="gpt-4o-mini",
    output_type=GapAnalysis,
)

In [ ]:
# 5. Writer Agent

writer_agent = Agent(
    name="Report Writer",
    instructions="""You are a senior career consultant writing a comprehensive CV deep research report.

You receive the CV text, ATS results, market research, company research, and gap analysis.

Produce a thorough, actionable report. Your rewrite suggestions must reference actual bullet points
from the CV and provide improved versions that are specific, quantified, and keyword-rich.

Your interview prep topics should be derived from the research, not generic.
Your 30-day action plan should be concrete and ordered by priority.""",
    model="gpt-4o-mini",
    output_type=FinalReport,
)

## The Deep Research Pipeline

Plan -> Search -> Analyze -> Write

In [ ]:
async def plan_research(target_role: str, company: str = "") -> ResearchPlan:
    """Use the planner agent to decide what to research."""
    print("Planning research...")
    query = f"Target role: {target_role}"
    if company:
        query += f"\nTarget company: {company}"
    result = await Runner.run(planner_agent, query)
    plan = result.final_output
    total = len(plan.role_searches) + len(plan.company_searches)
    print(f"Planned {total} searches ({len(plan.role_searches)} role + {len(plan.company_searches)} company)")
    return plan

In [ ]:
async def perform_search(item: ResearchItem) -> str:
    """Run a single web search via the search agent."""
    input_msg = f"Search term: {item.query}\nReason: {item.reason}"
    result = await Runner.run(search_agent, input_msg)
    return result.final_output


async def perform_all_searches(plan: ResearchPlan) -> dict:
    """Run all searches in parallel."""
    print("Searching the web...")

    role_tasks = [asyncio.create_task(perform_search(item)) for item in plan.role_searches]
    company_tasks = [asyncio.create_task(perform_search(item)) for item in plan.company_searches]

    role_results = await asyncio.gather(*role_tasks)
    company_results = await asyncio.gather(*company_tasks) if company_tasks else []

    print(f"Completed {len(role_results)} role searches + {len(company_results)} company searches")
    return {
        "role_research": "\n\n---\n\n".join(role_results),
        "company_research": "\n\n---\n\n".join(company_results) if company_results else "No company specified.",
    }

In [ ]:
async def review_cv(cv_file: str, job_keywords: str = "") -> str:
    """Run ATS review via the reviewer agent."""
    print("Running ATS review...")
    msg = f"Review the CV at '{cv_file}'."
    if job_keywords.strip():
        msg += f" Job keywords: {job_keywords}"
    result = await Runner.run(ats_reviewer_agent, msg)
    print("ATS review complete")
    return result.final_output

In [ ]:
async def analyze_gaps(cv_review: str, research: dict) -> GapAnalysis:
    """Run gap analysis by combining CV data with research."""
    print("Analyzing gaps...")
    input_msg = (
        f"## CV Review & ATS Results\n{cv_review}\n\n"
        f"## Market Research on Target Role\n{research['role_research']}\n\n"
        f"## Company Research\n{research['company_research']}"
    )
    result = await Runner.run(gap_analyst_agent, input_msg)
    print(f"Found {len(result.final_output.gaps)} gaps, fit score: {result.final_output.overall_fit_score}/100")
    return result.final_output

In [ ]:
async def write_final_report(cv_review: str, research: dict, gap_analysis: GapAnalysis) -> FinalReport:
    """Synthesize everything into the final report."""
    print("Writing final report...")
    input_msg = (
        f"## CV Review & ATS Results\n{cv_review}\n\n"
        f"## Market Research\n{research['role_research']}\n\n"
        f"## Company Research\n{research['company_research']}\n\n"
        f"## Gap Analysis\nFit score: {gap_analysis.overall_fit_score}/100\n"
        f"Strengths: {gap_analysis.strengths}\n"
        f"Gaps: {json.dumps([g.model_dump() for g in gap_analysis.gaps], indent=2)}"
    )
    result = await Runner.run(writer_agent, input_msg)
    print("Report complete")
    return result.final_output

## The Orchestrator Agent

A top-level orchestrator that ties it all together with the input guardrail.

In [ ]:
async def run_cv_deep_research(cv_file: str, target_role: str, company: str = "", job_keywords: str = ""):
    """Full pipeline: Plan → Search → Review → Analyze → Report."""

    with trace("CV Deep Research"):

        # Plan research and run ATS review in parallel
        plan_task = asyncio.create_task(plan_research(target_role, company))
        review_task = asyncio.create_task(review_cv(cv_file, job_keywords))
        research_plan, cv_review = await asyncio.gather(plan_task, review_task)

        # Execute all web searches in parallel
        research = await perform_all_searches(research_plan)

        # Gap analysis
        gap_analysis = await analyze_gaps(cv_review, research)

        # Final report
        report = await write_final_report(cv_review, research, gap_analysis)

    return report

## Run the Agent

In [ ]:
# Configure your inputs

CV_FILE = "Emmanuel_Samuel_CV_AI_DS_ML.pdf"  # Replace with your CV file path
TARGET_ROLE = "Machine Learning Engineer"
COMPANY = "Google"  # Optional: set to a company name like "Google" or leave empty
JOB_KEYWORDS = "python, machine learning, deep learning, MLOps, AWS, data pipelines, leadership"  # Comma-separated

In [ ]:
# Run with the input guardrail first

user_message = f"Review the CV at '{CV_FILE}' for the role of {TARGET_ROLE}."
if COMPANY:
    user_message += f" Target company: {COMPANY}."

# Validate input via guardrail
guardrail_agent = Agent(
    name="Input Gate",
    instructions="Pass through if input is valid.",
    model="gpt-4o-mini",
    input_guardrails=[validate_input],
)

try:
    await Runner.run(guardrail_agent, user_message)
    print("Input validated. Starting deep research...\n")
except Exception as e:
    print(f"Guardrail triggered: {e}")
    raise SystemExit("Please provide both a CV file path and a target role.")

In [ ]:
# Run the full deep research pipeline

report = await run_cv_deep_research(
    cv_file=CV_FILE,
    target_role=TARGET_ROLE,
    company=COMPANY,
    job_keywords=JOB_KEYWORDS,
)

print("\nDone!")

## View the Report

In [ ]:
# Build a clean markdown report

md = f"""# CV Deep Research Report

## Executive Summary
{report.executive_summary}

| Metric | Score |
|--------|-------|
| ATS Readiness | {report.ats_readiness_score}/100 |
| Role Fit | {report.role_fit_score}/100 |

## Market Insights
{report.market_insights}

## Company Insights
{report.company_insights}

## Gap Summary
{report.gap_summary}

## Rewrite Suggestions
"""

for i, s in enumerate(report.rewrite_suggestions, 1):
    md += f"\n### Suggestion {i}\n"
    md += f"Original: {s.original}\n\n"
    md += f"Improved: {s.improved}\n\n"
    md += f"*Why:* {s.reason}\n"

md += f"\n## Keywords to Add\n"
md += ", ".join(f"`{kw}`" for kw in report.keywords_to_add)

md += f"\n\n## Interview Prep Topics\n"
for topic in report.interview_prep_topics:
    md += f"- {topic}\n"

md += f"\n## 30-Day Action Plan\n"
for i, step in enumerate(report.thirty_day_action_plan, 1):
    md += f"{i}. {step}\n"

display(Markdown(md))

In [ ]:
# Save the report to a file (using the save_report tool to stay in the agent pattern)

save_result = save_report.on_invoke_tool(
    # We call the underlying function directly
    ctx=None,
    input=json.dumps({"filename": "cv_deep_research_report.md", "report_content": md}),
)
# Or just save it directly:
with open("cv_deep_research_report.md", "w", encoding="utf-8") as f:
    f.write(md)

print("Report saved to cv_deep_research_report.md")

## Gradio Interface

Upload a CV, enter a target role, and a company name to get a full deep research review.

In [ ]:
import gradio as gr

async def gradio_review(cv_file, target_role, company, job_kw):
    """Gradio wrapper for the deep research pipeline."""
    if not cv_file or not target_role.strip():
        return "Please provide both a CV file and a target role."

    report = await run_cv_deep_research(
        cv_file=cv_file.name,
        target_role=target_role.strip(),
        company=company.strip() if company else "",
        job_keywords=job_kw.strip() if job_kw else "",
    )

    # Format as readable text
    lines = []
    lines.append(f"EXECUTIVE SUMMARY\n{report.executive_summary}\n")
    lines.append(f"ATS Readiness: {report.ats_readiness_score}/100")
    lines.append(f"Role Fit: {report.role_fit_score}/100\n")
    lines.append(f"MARKET INSIGHTS\n{report.market_insights}\n")
    lines.append(f"COMPANY INSIGHTS\n{report.company_insights}\n")
    lines.append(f"GAP SUMMARY\n{report.gap_summary}\n")
    lines.append("REWRITE SUGGESTIONS")
    for i, s in enumerate(report.rewrite_suggestions, 1):
        lines.append(f"\n{i}. Original: {s.original}")
        lines.append(f"   Improved: {s.improved}")
        lines.append(f"   Why: {s.reason}")
    lines.append(f"\nKEYWORDS TO ADD\n{', '.join(report.keywords_to_add)}\n")
    lines.append("INTERVIEW PREP TOPICS")
    for topic in report.interview_prep_topics:
        lines.append(f"  - {topic}")
    lines.append("\n30-DAY ACTION PLAN")
    for i, step in enumerate(report.thirty_day_action_plan, 1):
        lines.append(f"  {i}. {step}")

    return "\n".join(lines)


with gr.Blocks(title="CV Deep Research Agent") as demo:
    gr.Markdown("# CV Deep Research Agent\nUpload your CV, enter a target role, and get research-backed career positioning feedback.")
    with gr.Row():
        cv_input = gr.File(label="Upload CV (PDF or TXT)", file_types=[".pdf", ".txt"])
        with gr.Column():
            role_input = gr.Textbox(label="Target Role", placeholder="e.g., Machine Learning Engineer")
            company_input = gr.Textbox(label="Target Company (optional)", placeholder="e.g., Google")
            kw_input = gr.Textbox(label="Job Keywords (optional, comma-separated)",
                                  placeholder="python, MLOps, AWS, leadership...")
    output = gr.Textbox(label="Deep Research Report", lines=30)
    btn = gr.Button("Research & Review My CV", variant="primary")
    btn.click(fn=gradio_review, inputs=[cv_input, role_input, company_input, kw_input], outputs=output)

demo.launch()